# LeetCode 2605 — Form Smallest Number From Two Digit Arrays bằng Python

Link đề: https://leetcode.com/problems/form-smallest-number-from-two-digit-arrays/ · Độ khó: **Easy** · Chủ đề: Array, Hash Table, Enumeration

Skeleton LeetCode:

```python
class Solution:
    def minNumber(self, nums1: List[int], nums2: List[int]) -> int:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: hai mảng các **chữ số khác nhau** (`1..9`); tìm số **nhỏ nhất** mà trong cách viết thập phân của nó có **ít nhất một chữ số của `nums1` và ít nhất một chữ số của `nums2`**.
2. Dùng `set`, phép **giao** `&` để tìm chữ số **chung**, và `min` để lấy chữ số nhỏ nhất.
3. Suy luận "đáp án chỉ có **1 hoặc 2 chữ số**": nếu có chữ số chung thì một chữ số là đủ; nếu không thì phải ghép hai chữ số (nhỏ trước, lớn sau).
4. Viết 4 cách: duyệt số `1, 2, 3, ...` tới khi thoả, liệt kê mọi số ứng viên, tập hợp + công thức, và **bitmask** (mỗi chữ số là một bit).
5. Kiểm chứng bằng cách thử **mọi cặp tập con** các chữ số nhỏ, và bằng test ngẫu nhiên có seed.
6. Tránh các lỗi quen thuộc: ghép hai chữ số nhỏ nhất mà không sắp xếp, quên chữ số chung, so sánh **chuỗi** thay vì số.

## 2. Đề bài & ví dụ

Cho hai mảng `nums1` và `nums2`, mỗi mảng gồm các chữ số **đôi một khác nhau**. Trả về **số nhỏ nhất** có chứa **ít nhất một chữ số từ mỗi mảng**.

| Ví dụ | nums1 | nums2 | Output | Giải thích |
|---|---|---|---|---|
| 1 | `[4,1,3]` | `[5,7]` | `15` | số `15` có chữ số `1` (từ `nums1`) và chữ số `5` (từ `nums2`); không có số nhỏ hơn thoả điều kiện |
| 2 | `[3,5,2,6]` | `[3,1,7,9,6]` | `3` | chữ số `3` nằm ở cả hai mảng nên số `3` (một chữ số) thoả |

Constraints: `1 <= nums1.length, nums2.length <= 9`; `1 <= nums1[i], nums2[i] <= 9`; các chữ số trong mỗi mảng là **duy nhất**.

In [1]:
nums1, nums2 = [4, 1, 3], [5, 7]
for candidate in (15, 51, 17, 13, 14):
    s = str(candidate)
    has1 = any(int(ch) in nums1 for ch in s)
    has2 = any(int(ch) in nums2 for ch in s)
    print(f"{candidate}: có chữ số của nums1? {has1} | của nums2? {has2} | thoả? {has1 and has2}")

15: có chữ số của nums1? True | của nums2? True | thoả? True
51: có chữ số của nums1? True | của nums2? True | thoả? True
17: có chữ số của nums1? True | của nums2? True | thoả? True
13: có chữ số của nums1? True | của nums2? False | thoả? False
14: có chữ số của nums1? True | của nums2? False | thoả? False


## 3. Kiến thức nền

### 3.1. `set` và phép giao `&`

`set` là tập hợp các giá trị **không trùng**. Phép giao `A & B` cho các phần tử **có ở cả hai** tập. Kiểm tra "A và B có chung phần tử không" chính là kiểm tra `A & B` có rỗng hay không (tập rỗng là "sai" trong `if`).

In [2]:
a = {4, 1, 3}
b = {5, 3, 7}
print(a & b)                 # {3}
print(a & {5, 7})            # set()  (rỗng)
if a & b:
    print("có chữ số chung")
if not (a & {5, 7}):
    print("không có chữ số chung")

{3}
set()
có chữ số chung
không có chữ số chung


### 3.2. `min`, `max` trên list/set

`min(x)` lấy phần tử nhỏ nhất của list hoặc set. `min(a, b)` và `max(a, b)` cho hai số. Dùng để sắp **chữ số nhỏ đứng trước** khi ghép số hai chữ số.

In [3]:
print(min([4, 1, 3]), min({5, 7}), min(7, 5), max(7, 5))

1 5 5 7


### 3.3. Ghép chữ số thành số: `10 * a + b`

Số hai chữ số có hàng chục `a` và hàng đơn vị `b` bằng `10 * a + b` (ví dụ `a = 1`, `b = 5` cho `15`). Với cùng hai chữ số `a < b`: `10 * a + b` **nhỏ hơn** `10 * b + a` — hàng chục nhỏ hơn cho số nhỏ hơn.

In [4]:
a, b = 1, 5
print(10 * a + b, 10 * b + a)      # 15 51
print(10 * min(a, b) + max(a, b))  # luôn là cách ghép nhỏ nhất

15 51
15


### 3.4. Chữ số của một số: `str(n)` và `any`

`str(15)` là `"15"`; duyệt ký tự rồi `int(ch)` ra từng chữ số. `any(điều_kiện for ... )` đúng nếu **ít nhất một** phần tử thoả. Ta dùng để hỏi "số này có chữ số nào thuộc `nums1`?".

In [5]:
nums1 = [4, 1, 3]
for n in (15, 7, 40):
    digits = [int(ch) for ch in str(n)]
    print(n, digits, "có chữ số thuộc nums1?", any(d in nums1 for d in digits))

15 [1, 5] có chữ số thuộc nums1? True
7 [7] có chữ số thuộc nums1? False
40 [4, 0] có chữ số thuộc nums1? True


### 3.5. Bitmask — mỗi chữ số là một bit

Ta có thể biểu diễn một **tập chữ số** bằng **một số nguyên**: bit thứ `d` bật nếu chữ số `d` thuộc tập. `1 << d` là số có đúng bit `d` bật. `mask |= 1 << d` thêm chữ số `d` vào tập. Giao hai tập là `mask1 & mask2`.

Muốn tìm chữ số **nhỏ nhất** trong mask: `(mask & -mask)` cô lập bit thấp nhất, `.bit_length() - 1` cho chỉ số của nó.

In [6]:
mask = 0
for d in [4, 1, 3]:
    mask |= 1 << d
print(bin(mask), mask)                      # bit 1, 3, 4 bật
low = mask & -mask                          # chỉ giữ bit thấp nhất
print(bin(low), "-> chữ số nhỏ nhất =", low.bit_length() - 1)

0b11010 26
0b10 -> chữ số nhỏ nhất = 1


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Số ta tạo cần **có** một chữ số của `nums1` và **có** một chữ số của `nums2`:

- Nếu **một chữ số vừa thuộc `nums1`, vừa thuộc `nums2`** thì **số một chữ số** đó đã thoả, và số một chữ số luôn nhỏ hơn mọi số hai chữ số. Trong các chữ số chung, ta chọn **nhỏ nhất**.
- Nếu **không có** chữ số chung thì cần ít nhất **hai chữ số**: một của `nums1`, một của `nums2`. Hai chữ số nhỏ nhất của mỗi bên là `min(nums1)` và `min(nums2)`; **chữ số nhỏ hơn** đứng hàng chục.

Ví dụ 1: `nums1 = [4, 1, 3]`, `nums2 = [5, 7]` → không chung; `min1 = 1`, `min2 = 5` → `15`.
Ví dụ 2: `nums1 = [3, 5, 2, 6]`, `nums2 = [3, 1, 7, 9, 6]` → chung `{3, 6}` → nhỏ nhất `3`.

In [7]:
for nums1, nums2 in ([4, 1, 3], [5, 7]), ([3, 5, 2, 6], [3, 1, 7, 9, 6]):
    common = set(nums1) & set(nums2)
    print("nums1 =", nums1, "nums2 =", nums2)
    print("  chữ số chung:", common)
    if common:
        print("  đáp án:", min(common))
    else:
        lo, hi = sorted([min(nums1), min(nums2)])
        print("  min1, min2 =", min(nums1), min(nums2), "-> đáp án:", 10 * lo + hi)

nums1 = [4, 1, 3] nums2 = [5, 7]
  chữ số chung: set()
  min1, min2 = 1 5 -> đáp án: 15
nums1 = [3, 5, 2, 6] nums2 = [3, 1, 7, 9, 6]
  chữ số chung: {3, 6}
  đáp án: 3


## 5. Cách 1 — Brute Force / baseline

Duyệt `n = 1, 2, 3, ...` theo thứ tự tăng dần, **số đầu tiên** thoả điều kiện là đáp án. Vì mọi chữ số đều thuộc `1..9`, số hai chữ số `10 * a + b` với `a` từ `nums1`, `b` từ `nums2` luôn tồn tại và `<= 99`, nên vòng lặp dừng trước `100`.

In [8]:
from typing import List

def min_number_scan(nums1: List[int], nums2: List[int]) -> int:
    set1, set2 = set(nums1), set(nums2)
    for n in range(1, 100):
        digits = {int(ch) for ch in str(n)}        # tập các chữ số của n
        if digits & set1 and digits & set2:        # có chữ số của nums1 VÀ có chữ số của nums2
            return n
    return -1                                      # không xảy ra với input hợp lệ

In [9]:
print(min_number_scan([4, 1, 3], [5, 7]))                 # 15
print(min_number_scan([3, 5, 2, 6], [3, 1, 7, 9, 6]))     # 3
print(min_number_scan([9], [1]))                          # 19

15
3
19


**Độ phức tạp:** duyệt tối đa 99 số, mỗi số xử lý tối đa 2 chữ số → Time `O(1)` theo hằng số (≈ `99 * 2` phép); nếu tính theo kích thước mảng thì mỗi lần kiểm tra giao tốn `O(len)`; Space `O(len(nums1) + len(nums2))`.

**Nút thắt:** ta thử nhiều số vô ích (ví dụ `1, 2, ..., 14` trong ví dụ 1). Chỉ cần lập luận "đáp án có 1 hoặc 2 chữ số" là tính thẳng được.

## 6. Ý tưởng thuật toán / bước đột phá

Đáp án luôn thuộc một trong hai dạng:

1. **Một chữ số** `d` với `d` thuộc **cả hai** mảng → đáp án là `min(chữ số chung)`.
2. Nếu không có chữ số chung → **hai chữ số** `x` (từ `nums1`) và `y` (từ `nums2`), sắp cho chữ số **nhỏ ở hàng chục**: `10 * min(x, y) + max(x, y)`, với `x = min(nums1)`, `y = min(nums2)` (chữ số nhỏ nhất ở mỗi mảng cho tổ hợp nhỏ nhất).

Quy tắc "nếu… thì…": `common` không rỗng → `min(common)`; ngược lại → ghép hai chữ số nhỏ nhất, nhỏ đứng trước.

In [10]:
def min_number_demo(nums1, nums2):
    common = set(nums1) & set(nums2)
    if common:
        return min(common)
    a, b = min(nums1), min(nums2)
    return min(a, b) * 10 + max(a, b)

print(min_number_demo([4, 1, 3], [5, 7]), min_number_demo([3, 5, 2, 6], [3, 1, 7, 9, 6]))

15 3


## 7. Các cách tiếp theo

Ta có thêm 3 cách: **(2)** liệt kê mọi số ứng viên rồi lấy `min`, **(3)** tập hợp + công thức (như mục 6), **(4)** bitmask.

### Cách 2 — Liệt kê mọi ứng viên rồi lấy `min`

Giải thích từng đoạn:

1. Mỗi chữ số chung `d` là một ứng viên `d`.
2. Mỗi cặp `(a, b)` với `a` từ `nums1`, `b` từ `nums2` cho hai ứng viên: `10 * a + b` và `10 * b + a`.
3. Đáp án là `min` của mọi ứng viên (kể cả ứng viên một chữ số). Cách này không cần lập luận "ghép nhỏ nhất" — nhưng vẫn dựa vào việc đáp án chỉ có 1 hoặc 2 chữ số.

In [11]:
def min_number_candidates(nums1, nums2):
    candidates = []
    for a in nums1:
        for b in nums2:
            if a == b:
                candidates.append(a)                # chữ số chung -> số một chữ số
            candidates.append(10 * a + b)           # a ở hàng chục
            candidates.append(10 * b + a)           # b ở hàng chục
    return min(candidates)

print(min_number_candidates([4, 1, 3], [5, 7]), min_number_candidates([3, 5, 2, 6], [3, 1, 7, 9, 6]))

15 3


In [12]:
from typing import List

class SolutionCandidates:
    def minNumber(self, nums1: List[int], nums2: List[int]) -> int:
        candidates = []
        for a in nums1:
            for b in nums2:
                if a == b:
                    candidates.append(a)
                candidates.append(10 * a + b)
                candidates.append(10 * b + a)
        return min(candidates)

Độ phức tạp Cách 2: Time `O(n * m)` với `n = len(nums1)`, `m = len(nums2)` (tối đa `9 * 9`); Space `O(n * m)` cho list ứng viên.

### Cách 3 — Tập hợp + công thức

Giao `set(nums1) & set(nums2)`; có thì `min(...)`; không thì ghép hai chữ số nhỏ nhất, chữ số nhỏ đứng trước.

In [13]:
def min_number_sets(nums1, nums2):
    common = set(nums1) & set(nums2)
    if common:
        return min(common)
    a, b = min(nums1), min(nums2)
    return min(a, b) * 10 + max(a, b)

print(min_number_sets([4, 1, 3], [5, 7]), min_number_sets([3, 5, 2, 6], [3, 1, 7, 9, 6]))

15 3


In [14]:
from typing import List

class SolutionSets:
    def minNumber(self, nums1: List[int], nums2: List[int]) -> int:
        common = set(nums1) & set(nums2)
        if common:
            return min(common)
        a, b = min(nums1), min(nums2)
        return min(a, b) * 10 + max(a, b)

Độ phức tạp Cách 3: Time `O(n + m)`; Space `O(n + m)` cho hai set.

### Cách 4 — Bitmask

Mỗi mảng thành một mask: `m1`, `m2`. Chữ số chung ↔ bit chung trong `m1 & m2`. Chữ số nhỏ nhất trong một mask `m` có chỉ số `(m & -m).bit_length() - 1` (bit thấp nhất). Không cần `set` hay `min` trên list.

In [15]:
def lowest_digit(mask):
    return (mask & -mask).bit_length() - 1

def min_number_bitmask(nums1, nums2):
    m1 = m2 = 0
    for d in nums1:
        m1 |= 1 << d
    for d in nums2:
        m2 |= 1 << d
    common = m1 & m2
    if common:
        return lowest_digit(common)
    a, b = lowest_digit(m1), lowest_digit(m2)
    return min(a, b) * 10 + max(a, b)

print(min_number_bitmask([4, 1, 3], [5, 7]), min_number_bitmask([3, 5, 2, 6], [3, 1, 7, 9, 6]))

15 3


In [16]:
from typing import List

class SolutionBitmask:
    def minNumber(self, nums1: List[int], nums2: List[int]) -> int:
        m1 = m2 = 0
        for d in nums1:
            m1 |= 1 << d
        for d in nums2:
            m2 |= 1 << d
        common = m1 & m2
        if common:
            return (common & -common).bit_length() - 1
        a = (m1 & -m1).bit_length() - 1
        b = (m2 & -m2).bit_length() - 1
        return min(a, b) * 10 + max(a, b)

Độ phức tạp Cách 4: Time `O(n + m)`; Space `O(1)` (hai số nguyên). Mask chỉ cần 10 bit (`0..9`), nên các phép `&`, `|` coi là `O(1)`.

## 8. TRACE từng bước

Trace Cách 4 (bitmask) với `nums1 = [4, 1, 3]`, `nums2 = [5, 7]`.

| bước | thao tác | giá trị |
|---|---|---|
| 1 | dựng `m1` từ `4, 1, 3` | bit 1, 3, 4 bật → `0b11010 = 26` |
| 2 | dựng `m2` từ `5, 7` | bit 5, 7 bật → `0b10100000 = 160` |
| 3 | `common = m1 & m2` | `0` → không có chữ số chung |
| 4 | `a = lowest_digit(m1)` | bit thấp nhất của `26` là bit 1 → `1` |
| 5 | `b = lowest_digit(m2)` | bit thấp nhất của `160` là bit 5 → `5` |
| 6 | `min(a, b) * 10 + max(a, b)` | `1 * 10 + 5 = 15` |

In [17]:
def trace_bitmask(nums1, nums2):
    m1 = m2 = 0
    for d in nums1:
        m1 |= 1 << d
    for d in nums2:
        m2 |= 1 << d
    print(f"m1 = {m1:#b} = {m1}   (các chữ số {nums1})")
    print(f"m2 = {m2:#b} = {m2}   (các chữ số {nums2})")
    common = m1 & m2
    print(f"common = m1 & m2 = {common:#b} = {common}")
    if common:
        ans = (common & -common).bit_length() - 1
        print("có chữ số chung; chữ số chung nhỏ nhất =", ans)
        return ans
    a = (m1 & -m1).bit_length() - 1
    b = (m2 & -m2).bit_length() - 1
    print(f"không chung. chữ số nhỏ nhất mỗi bên: a={a}, b={b}")
    ans = min(a, b) * 10 + max(a, b)
    print(f"ghép: nhỏ đứng trước -> {ans}")
    return ans

trace_bitmask([4, 1, 3], [5, 7])
trace_bitmask([3, 5, 2, 6], [3, 1, 7, 9, 6])

m1 = 0b11010 = 26   (các chữ số [4, 1, 3])
m2 = 0b10100000 = 160   (các chữ số [5, 7])
common = m1 & m2 = 0b0 = 0
không chung. chữ số nhỏ nhất mỗi bên: a=1, b=5
ghép: nhỏ đứng trước -> 15
m1 = 0b1101100 = 108   (các chữ số [3, 5, 2, 6])
m2 = 0b1011001010 = 714   (các chữ số [3, 1, 7, 9, 6])
common = m1 & m2 = 0b1001000 = 72
có chữ số chung; chữ số chung nhỏ nhất = 3


3

In [18]:
def trace_sets(nums1, nums2):
    set1, set2 = set(nums1), set(nums2)
    common = set1 & set2
    print(f"nums1={sorted(set1)}, nums2={sorted(set2)}, chung={sorted(common)}")
    if common:
        print("-> có chữ số chung, đáp án là chữ số chung nhỏ nhất:", min(common))
        return min(common)
    a, b = min(set1), min(set2)
    print(f"-> không chung: min1={a}, min2={b}, hàng chục={min(a, b)}, hàng đơn vị={max(a, b)}")
    return 10 * min(a, b) + max(a, b)

trace_sets([9], [1])       # nhỏ phải đứng trước: 19, không phải 91
trace_sets([2, 5], [5, 8])

nums1=[9], nums2=[1], chung=[]
-> không chung: min1=9, min2=1, hàng chục=1, hàng đơn vị=9
nums1=[2, 5], nums2=[5, 8], chung=[5]
-> có chữ số chung, đáp án là chữ số chung nhỏ nhất: 5


5

## 9. Code nộp LeetCode

Nộp Cách 3 (tập hợp + công thức): ngắn, dễ giải thích và đủ nhanh.

In [19]:
from typing import List

class Solution:
    def minNumber(self, nums1: List[int], nums2: List[int]) -> int:
        common = set(nums1) & set(nums2)
        if common:
            return min(common)                       # một chữ số chung nhỏ nhất
        a, b = min(nums1), min(nums2)
        return min(a, b) * 10 + max(a, b)            # hai chữ số, nhỏ đứng trước

print(Solution().minNumber([4, 1, 3], [5, 7]))                 # 15
print(Solution().minNumber([3, 5, 2, 6], [3, 1, 7, 9, 6]))     # 3

15
3


## 10. Vì sao thuật toán đúng

- **Đáp án có tối đa 2 chữ số:** với bất kỳ `a` trong `nums1` và `b` trong `nums2`, số `10 * a + b` có đủ cả hai chữ số và `<= 99`. Vậy đáp án `<= 99`.
- **Có chữ số chung:** nếu `d` thuộc cả hai mảng thì số `d` (một chữ số) thoả, và mọi số một chữ số nhỏ hơn thoả đều phải là một chữ số chung `d' < d`. Vì ta lấy `min(common)`, kết quả là số một chữ số thoả nhỏ nhất. Mọi số hai chữ số `>= 10 > min(common)`, nên không thể tốt hơn.
- **Không có chữ số chung:** không có số một chữ số nào thoả (một chữ số duy nhất không thể vừa thuộc `nums1` vừa thuộc `nums2`). Số hai chữ số thoả phải chứa một chữ số `x` của `nums1` và một chữ số `y` của `nums2` (khác nhau vì không chung). Có hai cách ghép `10x + y` và `10y + x`; số nhỏ hơn là ghép chữ số **nhỏ ở hàng chục**: `10 * min(x, y) + max(x, y)`. Hàm này tăng theo cả `x` lẫn `y` nên chọn `x = min(nums1)`, `y = min(nums2)` cho giá trị nhỏ nhất.
- **Cách bitmask:** bit thấp nhất của một mask tương ứng chữ số nhỏ nhất của tập, vì chỉ số bit chính là chữ số; `m & -m` cô lập đúng bit thấp nhất.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(nums1)`, `m = len(nums2)` (đều `<= 9`).

| Cách | Time | Space |
|---|---|---|
| 1. Duyệt `n = 1..99` | `O(99 * (n + m))` ≈ hằng số | `O(n + m)` |
| 2. Liệt kê ứng viên | `O(n * m)` | `O(n * m)` |
| 3. Tập hợp + công thức | `O(n + m)` | `O(n + m)` |
| 4. Bitmask | `O(n + m)` | `O(1)` |

Mọi cách chạy tức thì vì dữ liệu cực nhỏ; điều quan trọng là **lập luận đáp án tối đa hai chữ số**.

## 12. Bộ test

Test 1: ví dụ đề và edge case: một chữ số mỗi mảng (giống nhau / khác nhau theo cả hai thứ tự), nhiều chữ số chung, mảng đầy đủ `1..9`.

In [20]:
impls = [
    ("scan", min_number_scan),
    ("candidates", min_number_candidates),
    ("sets", min_number_sets),
    ("bitmask", min_number_bitmask),
    ("class_candidates", SolutionCandidates().minNumber),
    ("class_sets", SolutionSets().minNumber),
    ("class_bitmask", SolutionBitmask().minNumber),
    ("submit", Solution().minNumber),
]
tests = [
    ([4, 1, 3], [5, 7], 15),
    ([3, 5, 2, 6], [3, 1, 7, 9, 6], 3),
    ([5], [5], 5),                         # một chữ số giống nhau
    ([9], [1], 19),                        # khác nhau: nhỏ đứng trước
    ([1], [9], 19),                        # đổi chỗ hai mảng: vẫn 19
    ([5, 8], [2, 8], 8),                   # chung 8 -> 8 < 25
    ([1, 5], [2, 5], 5),                   # chung 5 -> 5 < 12
    ([2, 9], [3, 8], 23),                  # không chung: 23
    ([1, 2, 3, 4, 5, 6, 7, 8, 9], [1, 2, 3, 4, 5, 6, 7, 8, 9], 1),
    ([9, 8, 7, 6, 5, 4, 3, 2], [1], 12),   # mảng dài vs một chữ số, không chung
    ([6, 7], [8, 9, 6], 6),
]
for name, f in impls:
    for a, b, expected in tests:
        got = f(list(a), list(b))
        assert got == expected, (name, a, b, got, expected)
    print(name, "PASS", len(tests), "test")

scan PASS 11 test
candidates PASS 11 test
sets PASS 11 test
bitmask PASS 11 test
class_candidates PASS 11 test
class_sets PASS 11 test
class_bitmask PASS 11 test
submit PASS 11 test


Test 2: **vét cạn mọi cặp tập con** của các chữ số `1..6` (mỗi tập không rỗng; thứ tự trong mảng không ảnh hưởng đáp án), so với `min_number_scan` làm oracle.

In [21]:
from itertools import combinations

def all_subsets(universe):
    subsets = []
    for size in range(1, len(universe) + 1):
        for comb in combinations(universe, size):
            subsets.append(list(comb))
    return subsets

subs = all_subsets([1, 2, 3, 4, 5, 6])
count = 0
for a in subs:
    for b in subs:
        expected = min_number_scan(a, b)
        for name, f in impls:
            assert f(list(a), list(b)) == expected, (name, a, b)
        count += 1
print("đã so khớp", count, "cặp tập con (mỗi tập là một mảng hợp lệ)")

đã so khớp 3969 cặp tập con (mỗi tập là một mảng hợp lệ)


Test 3: ngẫu nhiên có seed, **đúng constraints** (mỗi mảng là các chữ số `1..9` khác nhau, độ dài `1..9`, thứ tự xáo trộn); oracle độc lập: thử mọi số `1..99`.

In [22]:
import random
random.seed(2605)

def oracle(nums1, nums2):
    for n in range(1, 100):
        s = str(n)
        if any(ch in map(str, nums1) for ch in s) and any(ch in map(str, nums2) for ch in s):
            return n

for _ in range(3000):
    a = random.sample(range(1, 10), random.randint(1, 9))
    b = random.sample(range(1, 10), random.randint(1, 9))
    expected = oracle(a, b)
    for name, f in impls:
        assert f(list(a), list(b)) == expected, (name, a, b)
print("3000 test ngẫu nhiên: tất cả cách đều khớp oracle")

3000 test ngẫu nhiên: tất cả cách đều khớp oracle


Test 4: tính chất — hoán đổi hai mảng không đổi đáp án; kết quả `<= 99`; kết quả một chữ số ⇔ hai mảng có chữ số chung; hàm không sửa input.

In [23]:
random.seed(8)
for _ in range(1000):
    a = random.sample(range(1, 10), random.randint(1, 9))
    b = random.sample(range(1, 10), random.randint(1, 9))
    keep_a, keep_b = list(a), list(b)
    ans = Solution().minNumber(a, b)
    assert ans == Solution().minNumber(b, a)
    assert ans <= 99
    assert (ans < 10) == bool(set(a) & set(b))
    assert a == keep_a and b == keep_b
print("tính chất đối xứng, ngưỡng, tương đương 'có chữ số chung', không sửa input: PASS")

tính chất đối xứng, ngưỡng, tương đương 'có chữ số chung', không sửa input: PASS


## 13. Các lỗi người mới hay gặp

### Lỗi 1: ghép hai chữ số nhỏ nhất **không sắp xếp** (luôn `min1 * 10 + min2`)

In [24]:
def wrong_no_order(nums1, nums2):
    common = set(nums1) & set(nums2)
    if common:
        return min(common)
    return min(nums1) * 10 + min(nums2)       # SAI: nếu min(nums1) > min(nums2) thì số không nhỏ nhất

print(wrong_no_order([4, 1, 3], [5, 7]), "vs đúng", Solution().minNumber([4, 1, 3], [5, 7]))
print(wrong_no_order([9], [1]), "vs đúng", Solution().minNumber([9], [1]))

15 vs đúng 15
91 vs đúng 19


Vì sao sai: với `nums1 = [9]`, `nums2 = [1]` code ra `91`, trong khi `19` nhỏ hơn. Chữ số nhỏ hơn phải ở hàng **chục** → `min(a, b) * 10 + max(a, b)`.

### Lỗi 2: bỏ qua chữ số chung — luôn ghép hai chữ số

In [25]:
def wrong_ignore_common(nums1, nums2):
    a, b = min(nums1), min(nums2)
    return min(a, b) * 10 + max(a, b)          # SAI: chưa kiểm tra chữ số chung

print(wrong_ignore_common([1, 5], [2, 5]), "vs đúng", Solution().minNumber([1, 5], [2, 5]))
print(wrong_ignore_common([3, 5, 2, 6], [3, 1, 7, 9, 6]), "vs đúng", Solution().minNumber([3, 5, 2, 6], [3, 1, 7, 9, 6]))

12 vs đúng 5
12 vs đúng 3


Vì sao sai: nếu có chữ số chung `d` thì số `d` một chữ số thoả cả hai mảng — nhỏ hơn mọi số hai chữ số. Ví dụ `[1, 5]` và `[2, 5]` cho `5`, không phải `12`.

### Lỗi 3: chỉ kiểm tra chữ số chung nhỏ nhất của **mảng này** thay vì giao

In [26]:
def wrong_first_digit_only(nums1, nums2):
    if min(nums1) in nums2:                    # SAI: chữ số chung nhỏ nhất có thể không phải min(nums1)
        return min(nums1)
    a, b = min(nums1), min(nums2)
    return min(a, b) * 10 + max(a, b)

print(wrong_first_digit_only([1, 5], [2, 5]), "vs đúng", Solution().minNumber([1, 5], [2, 5]))

12 vs đúng 5


Vì sao sai: chữ số chung không nhất thiết là chữ số nhỏ nhất của một mảng. Hãy **tính giao** rồi lấy `min` của giao.

### Lỗi 4: so sánh **chuỗi** thay vì số

In [27]:
candidates = ["15", "3", "27"]
print(min(candidates), "<- so sánh theo thứ tự từ điển: '15' < '27' < '3'")
print(min(int(c) for c in candidates), "<- so sánh số đúng: 3")

15 <- so sánh theo thứ tự từ điển: '15' < '27' < '3'
3 <- so sánh số đúng: 3


Vì sao sai: `"15" < "3"` vì ký tự `'1' < '3'`, dù `15 > 3`. Khi nối chữ số bằng chuỗi (ví dụ `str(a) + str(b)`), phải đổi sang `int` **trước khi** so sánh.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Số nhỏ nhất chứa chữ số từ **mỗi** mảng" → số **ít chữ số hơn thì nhỏ hơn**: tìm chữ số **chung** trước (số một chữ số), rồi mới tới ghép hai chữ số.
- Ghép hai chữ số nhỏ nhất → luôn viết `min(a, b) * 10 + max(a, b)`.
- Với dữ liệu cực nhỏ, bạn có thể **vét cạn** (thử `1..99`) để kiểm chứng công thức — thói quen rất đáng có.
- Tập các chữ số `0..9` rất hợp với **bitmask** (10 bit); giao tập ↔ `&`, hợp tập ↔ `|`.
- Bài họ hàng: "Intersection of Two Arrays" (349), "Find the Difference of Two Arrays" (2215), "Minimum Common Value" (2540).

## 15. Biến thể của bài

1. Có chữ số `0` trong mảng: `0` không được đứng đầu một số nhiều chữ số — phải xử lý riêng.
2. Tạo số **lớn nhất** chứa ít nhất một chữ số từ mỗi mảng, với số chữ số tối đa cho trước.
3. Ba mảng (hoặc `k` mảng) — bài tập 5: đáp án có thể tới `k` chữ số.
4. Số được tạo phải dùng **mọi** chữ số của một mảng (bài khó hơn, quay về hoán vị).
5. Số phải chứa **đúng một** chữ số từ mỗi mảng (cấm chứa hai chữ số cùng mảng).
6. Hai mảng rất lớn (`10^5` chữ số nhưng chỉ `0..9`): đếm chữ số bằng `bitmask` vẫn `O(n)`.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Duyệt `1..99` | số đầu tiên thoả điều kiện | `O(99 * (n+m))` | `O(n+m)` | oracle, không cần lập luận |
| 2. Liệt kê ứng viên | `d`, `10a+b`, `10b+a`, rồi `min` | `O(n * m)` | `O(n * m)` | khi chưa chắc công thức |
| 3. Set + công thức | `min(common)` hoặc ghép `min1, min2` | `O(n + m)` | `O(n + m)` | cách nộp, dễ đọc |
| 4. Bitmask | mỗi chữ số một bit, `m & -m` | `O(n + m)` | `O(1)` | luyện tư duy bit, dữ liệu chữ số |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `Solution().minNumber([2, 6], [4, 6])` bằng bao nhiêu?

**Bài 2 (tính tay):** `nums1 = [3, 9]`, `nums2 = [8, 4]`. Có chữ số chung không? Ghép như thế nào, đáp án là gì?

**Bài 3 (điền TODO):** hoàn thành `common_digits(nums1, nums2)` trả về **list đã sắp tăng dần** các chữ số có ở cả hai mảng.

**Bài 4 (tự viết lại):** viết `min_number_mask(nums1, nums2)` bằng bitmask (không dùng `set`) mà không nhìn lại code.

**Bài 5 (mini challenge):** viết `min_number_three(a, b, c)` tìm **số nhỏ nhất** chứa ít nhất một chữ số từ **mỗi** trong ba mảng (đáp án tối đa 3 chữ số).

In [28]:
# Bài 1: đoán output rồi chạy để kiểm tra
print(Solution().minNumber([2, 6], [4, 6]))

# Bài 3: TODO (KHÔNG gọi hàm này khi chạy notebook)
def common_digits(nums1, nums2):
    result = []
    # TODO: với mỗi chữ số d trong nums1, nếu d cũng có trong nums2 thì thêm vào result
    # TODO: trả về result đã sắp tăng dần (dùng sorted)
    return result

6


## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):** có chữ số chung → số một chữ số; không có → ghép hai chữ số, nhỏ đứng trước.

**Mức 2 (vừa):** Bài 3: `sorted(set(nums1) & set(nums2))`. Bài 4: `m = 0; m |= 1 << d`; chữ số nhỏ nhất của mask `m` là `(m & -m).bit_length() - 1`. Bài 5: thử các số `1, 2, 3, ...` tới khi có chữ số của cả ba mảng, hoặc liệt kê mọi tổ hợp một chữ số từ mỗi mảng.

**Mức 3 (gần đáp án):** Bài 1: chữ số chung là `6` ⇒ `6`. Bài 2: không chung; `min1 = 3`, `min2 = 4` ⇒ `34`.

## 19. Đáp án tham khảo cho bài tập

In [29]:
# Bài 1: chung {6} -> 6
assert Solution().minNumber([2, 6], [4, 6]) == 6

# Bài 2: không chung; min1 = 3, min2 = 4 -> 34
assert Solution().minNumber([3, 9], [8, 4]) == 34

# Bài 3
def common_digits_answer(nums1, nums2):
    return sorted(set(nums1) & set(nums2))

assert common_digits_answer([4, 1, 3], [5, 3, 1]) == [1, 3]
assert common_digits_answer([4, 1, 3], [5, 7]) == []

# Bài 4
def min_number_mask(nums1, nums2):
    m1 = m2 = 0
    for d in nums1:
        m1 |= 1 << d
    for d in nums2:
        m2 |= 1 << d
    common = m1 & m2
    if common:
        return (common & -common).bit_length() - 1
    a = (m1 & -m1).bit_length() - 1
    b = (m2 & -m2).bit_length() - 1
    return min(a, b) * 10 + max(a, b)

for a in subs:
    for b in subs:
        assert min_number_mask(a, b) == Solution().minNumber(a, b)

# Bài 5
def min_number_three(a, b, c):
    sa, sb, sc = set(a), set(b), set(c)
    for n in range(1, 1000):
        digits = {int(ch) for ch in str(n)}
        if digits & sa and digits & sb and digits & sc:
            return n

assert min_number_three([1], [2], [3]) == 123
assert min_number_three([1, 2], [2, 5], [2, 7]) == 2
assert min_number_three([5, 1], [5, 2], [3]) == 35
# đối chiếu với bài hai mảng khi mảng thứ ba chứa mọi chữ số
full = list(range(1, 10))
for a in subs[:30]:
    for b in subs[:30]:
        assert min_number_three(a, b, full) == Solution().minNumber(a, b)
print("tất cả đáp án bài tập: PASS")

tất cả đáp án bài tập: PASS


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "giao tập, rồi ghép hai chữ số":**

```python
common = set(nums1) & set(nums2)
if common:
    return min(common)                          # số một chữ số
a, b = min(nums1), min(nums2)
return min(a, b) * 10 + max(a, b)               # hai chữ số, nhỏ đứng trước
```

**Checklist trước khi submit:**

- [ ] Đã kiểm tra **chữ số chung** trước khi ghép hai chữ số?
- [ ] Khi ghép, chữ số **nhỏ hơn** đứng hàng chục (`min(a, b) * 10 + max(a, b)`)?
- [ ] Lấy `min` của **giao** (không phải `min` của một mảng)?
- [ ] Trả về `int`, không so sánh chuỗi?
- [ ] Đã thử mảng một chữ số (giống nhau / khác nhau), mảng đủ `1..9`?
- [ ] Đã vét cạn `1..99` để đối chiếu?
- [ ] Chữ ký đúng `minNumber(self, nums1: List[int], nums2: List[int]) -> int`?